ChatGPT advised that the primary prediction unit is O*NET-SOC Code. I will look for CSVs containing those first.

In [31]:
import pandas as pd

In [32]:
df_occupation_data = pd.read_csv('ONET data/Occupation Data.csv')
df_task_statements = pd.read_csv('ONET data/Task Statements.csv')
df_skills = pd.read_csv('ONET data/Skills.csv')
df_abilities = pd.read_csv('ONET data/Abilities.csv')
df_work_activities = pd.read_csv('ONET data/Work Activities.csv')
df_knowledge = pd.read_csv('ONET data/Knowledge.csv')
df_work_context = pd.read_csv('ONET data/Work Context.csv')
df_tools_used = pd.read_csv('ONET data/Tools Used.csv')
df_technology_skills = pd.read_csv('ONET data/Technology Skills.csv')

In [33]:
dfs = {
    "occupation_data": df_occupation_data,
    "task_statements": df_task_statements,
    "skills": df_skills,
    "abilities": df_abilities,
    "work_activities": df_work_activities,
    "knowledge": df_knowledge,
    "work_context": df_work_context,
    "tools_used": df_tools_used,
    "technology_skills": df_technology_skills
}

In [34]:
df_task_statements

,O*NET-SOC Code,Title,Task ID,Task,Task Type,Incumbents Responding,Date,Domain Source
0,11-1011.00,Chief Executives,8823,Direct or coordinate an organization's financi...,Core,95.0,08/2023,Incumbent
1,11-1011.00,Chief Executives,8824,"Confer with board members, organization offici...",Core,95.0,08/2023,Incumbent
2,11-1011.00,Chief Executives,8827,"Prepare budgets for approval, including those ...",Core,95.0,08/2023,Incumbent
3,11-1011.00,Chief Executives,8826,"Direct, plan, or implement policies, objective...",Core,94.0,08/2023,Incumbent
4,11-1011.00,Chief Executives,8834,Prepare or present reports concerning activiti...,Core,95.0,08/2023,Incumbent
...,...,...,...,...,...,...,...,...
18791,53-7121.00,"Tank Car, Truck, and Ship Loaders",12807,Unload cars containing liquids by connecting h...,Supplemental,85.0,08/2019,Incumbent
18792,53-7121.00,"Tank Car, Truck, and Ship Loaders",12804,"Clean interiors of tank cars or tank trucks, u...",Supplemental,85.0,08/2019,Incumbent
18793,53-7121.00,"Tank Car, Truck, and Ship Loaders",12803,Lower gauge rods into tanks or read meters to ...,Supplemental,85.0,08/2019,Incumbent
18794,53-7121.00,"Tank Car, Truck, and Ship Loaders",12805,Operate conveyors and equipment to transfer gr...,Supplemental,85.0,08/2019,Incumbent


In [35]:
df_occupation_data['O*NET-SOC Code'].nunique()

1016

In [36]:
df_task_statements['O*NET-SOC Code'].nunique()

923

In [37]:
df_skills['O*NET-SOC Code'].nunique()

894

In [38]:
df_abilities['O*NET-SOC Code'].nunique()

894

In [39]:
df_work_activities['O*NET-SOC Code'].nunique()

894

In [40]:
df_knowledge['O*NET-SOC Code'].nunique()

894

In [41]:
df_work_context['O*NET-SOC Code'].nunique()

894

In [42]:
df_tools_used['O*NET-SOC Code'].nunique()

902

In [43]:
df_technology_skills['O*NET-SOC Code'].nunique()

923

# Find Missing

In [44]:
soc_sets = {
    name: set(df['O*NET-SOC Code'].unique())
    for name, df in dfs.items()
}


In [45]:
all_socs = sorted(set.union(*soc_sets.values()))


In [46]:
import pandas as pd

coverage_df = pd.DataFrame({
    name: [soc in soc_sets[name] for soc in all_socs]
    for name in soc_sets
}, index=all_socs)

coverage_df.index.name = 'O*NET-SOC Code'
coverage_df = coverage_df.astype(int)

coverage_df['MISSING_COUNT'] = (
    len(coverage_df.columns) - coverage_df.sum(axis=1)
)

coverage_df

,occupation_data,task_statements,skills,abilities,work_activities,knowledge,work_context,tools_used,technology_skills,MISSING_COUNT
O*NET-SOC Code,,,,,,,,,,
11-1011.00,1,1,1,1,1,1,1,1,1,0
11-1011.03,1,1,1,1,1,1,1,1,1,0
11-1021.00,1,1,1,1,1,1,1,1,1,0
11-1031.00,1,1,0,0,0,0,0,1,1,5
11-2011.00,1,1,1,1,1,1,1,1,1,0
...,...,...,...,...,...,...,...,...,...,...
55-3014.00,1,0,0,0,0,0,0,0,0,8
55-3015.00,1,0,0,0,0,0,0,0,0,8
55-3016.00,1,0,0,0,0,0,0,0,0,8


In [47]:
coverage_df['MISSING_COUNT'].value_counts()

MISSING_COUNT
0    887
8     93
5     15
6     14
1      7
Name: count, dtype: int64

In [48]:
data_columns = coverage_df.columns.drop('MISSING_COUNT')

total_socs = len(coverage_df)

for col in data_columns:
    missing = (coverage_df[col] == 0).sum()
    print(f"{col}: {missing} jobs missing out of {total_socs}")

occupation_data: 0 jobs missing out of 1016
task_statements: 93 jobs missing out of 1016
skills: 122 jobs missing out of 1016
abilities: 122 jobs missing out of 1016
work_activities: 122 jobs missing out of 1016
knowledge: 122 jobs missing out of 1016
work_context: 122 jobs missing out of 1016
tools_used: 114 jobs missing out of 1016
technology_skills: 93 jobs missing out of 1016


# Narrowing Focus - Technology.

In [50]:
df_technology_skills.columns

Index(['O*NET-SOC Code', 'Title', 'Example', 'Commodity Code',
       'Commodity Title', 'Hot Technology', 'In Demand'],
      dtype='object')

In [63]:
df_technology_skills

,O*NET-SOC Code,Title,Example,Commodity Code,Commodity Title,Hot Technology,In Demand
0,11-1011.00,Chief Executives,Adobe Acrobat,43232202,Document management software,Y,N
1,11-1011.00,Chief Executives,AdSense Tracker,43232306,Data base user interface and query software,N,N
2,11-1011.00,Chief Executives,Atlassian JIRA,43232201,Content workflow software,Y,N
3,11-1011.00,Chief Executives,Blackbaud The Raiser's Edge,43232303,Customer relationship management CRM software,N,N
4,11-1011.00,Chief Executives,ComputerEase construction accounting software,43231601,Accounting software,N,N
...,...,...,...,...,...,...,...
32768,53-7121.00,"Tank Car, Truck, and Ship Loaders",Linux,43233004,Operating system software,Y,N
32769,53-7121.00,"Tank Car, Truck, and Ship Loaders",Microsoft Excel,43232110,Spreadsheet software,Y,N
32770,53-7121.00,"Tank Car, Truck, and Ship Loaders",Microsoft Office software,43231513,Office suite software,Y,N
32771,53-7121.00,"Tank Car, Truck, and Ship Loaders",SAP software,43231602,Enterprise resource planning ERP software,Y,N


In [51]:
df_technology_skills['O*NET-SOC Code'].nunique()

923

In [52]:
df_technology_skills['Title'].nunique()

923

In [53]:
df_technology_skills['Example'].nunique()

8785

In [54]:
df_technology_skills['Commodity Code'].nunique()

137

In [55]:
df_technology_skills['Commodity Title'].nunique()

137

In [ ]:
total_skills_by_title = (
    df_technology_skills
    .groupby(['O*NET-SOC Code', 'Title'])['Example']
    .nunique()
    .reset_index(name='TOTAL_EXAMPLES')
    .sort_values('TOTAL_EXAMPLES', ascending=False)
)

total_commods_by_title = (
    df_technology_skills
    .groupby(['O*NET-SOC Code', 'Title'])['Commodity Code']
    .nunique()
    .reset_index(name='TOTAL_COMMODITIES')
    .sort_values('TOTAL_COMMODITIES', ascending=False)
)

titles_skills_and_commods = pd.merge(total_skills_by_title, total_commods_by_title, on=['O*NET-SOC Code', 'Title'], how='inner')

examples_by_title = (
    df_technology_skills
    .groupby('Title')['Example']
    .apply(lambda x: '; '.join(
        f'"{e}"' for e in sorted(x.unique())
    ))
    .reset_index(name='example_list')
)

In [64]:
total_skills_by_title = (
    df_technology_skills
    .groupby(['O*NET-SOC Code', 'Title'])['Example']
    .nunique()
    .reset_index(name='TOTAL_EXAMPLES')
    .sort_values('TOTAL_EXAMPLES', ascending=False)
)

#total_skills_by_title

In [65]:
total_commods_by_title = (
    df_technology_skills
    .groupby(['O*NET-SOC Code', 'Title'])['Commodity Code']
    .nunique()
    .reset_index(name='TOTAL_COMMODITIES')
    .sort_values('TOTAL_COMMODITIES', ascending=False)
)

#total_commods_by_title

In [66]:
titles_skills_and_commods = pd.merge(total_skills_by_title, total_commods_by_title, on=['O*NET-SOC Code', 'Title'], how='inner')
titles_skills_and_commods

,O*NET-SOC Code,Title,TOTAL_EXAMPLES,TOTAL_COMMODITIES
0,15-1252.00,Software Developers,429,67
1,15-1253.00,Software Quality Assurance Analysts and Testers,428,68
2,15-1299.09,Information Technology Project Managers,334,62
3,15-1243.00,Database Architects,324,59
4,15-1211.00,Computer Systems Analysts,313,68
...,...,...,...,...
918,35-9021.00,Dishwashers,2,2
919,53-7041.00,Hoist and Winch Operators,2,2
920,53-4041.00,Subway and Streetcar Operators,2,2
921,51-4023.00,"Rolling Machine Setters, Operators, and Tender...",2,2


In [75]:
examples_by_title = (
    df_technology_skills
    .groupby('Title')['Example']
    .apply(lambda x: '; '.join(
        f'"{e}"' for e in sorted(x.unique())
    ))
    .reset_index(name='example_list')
)

#examples_by_title

In [73]:
titles_skills_and_commods = titles_skills_and_commods.merge(
    examples_by_title,
    on='Title',
    how='left'
)


In [74]:
titles_skills_and_commods

,O*NET-SOC Code,Title,TOTAL_EXAMPLES,TOTAL_COMMODITIES,example_list
0,15-1252.00,Software Developers,429,67,"""3M Post-it App""; ""A programming language APL""..."
1,15-1253.00,Software Quality Assurance Analysts and Testers,428,68,"""3M Post-it App""; ""A programming language APL""..."
2,15-1299.09,Information Technology Project Managers,334,62,"""24SevenOffice Project""; ""3M Post-it App""; ""AE..."
3,15-1243.00,Database Architects,324,59,"""3M Post-it App""; ""ADO.NET""; ""AJAX""; ""ASG Tech..."
4,15-1211.00,Computer Systems Analysts,313,68,"""3M Post-it App""; ""ADP Workforce Now""; ""AJAX"";..."
...,...,...,...,...,...
918,35-9021.00,Dishwashers,2,2,"""Facebook""; ""Microsoft Windows"""
919,53-7041.00,Hoist and Winch Operators,2,2,"""Microsoft Excel""; ""Microsoft Word"""
920,53-4041.00,Subway and Streetcar Operators,2,2,"""Microsoft Office software""; ""Word processing ..."
921,51-4023.00,"Rolling Machine Setters, Operators, and Tender...",2,2,"""Email software""; ""Web browser software"""


In [58]:
soc_commodity_breakdown = (
    df_technology_skills
    .groupby([
        'O*NET-SOC Code', 'Title'
    ])
    .agg(
        num_examples=('Example', 'nunique'),
        example_list=('Example',
                      lambda x: '; '.join(
                          f'"{e}"' for e in sorted(x.unique())
                      ))
    )
    .reset_index()
)

soc_commodity_breakdown

,O*NET-SOC Code,Title,num_examples,example_list
0,11-1011.00,Chief Executives,49,"""AdSense Tracker""; ""Adobe Acrobat""; ""Atlassian..."
1,11-1011.03,Chief Sustainability Officers,21,"""Adobe Acrobat""; ""Adobe Photoshop""; ""Email sof..."
2,11-1021.00,General and Operations Managers,146,"""ADP Workforce Now""; ""AMG Teleran SalesInSync""..."
3,11-1031.00,Legislators,32,"""Adobe Acrobat""; ""Adobe FrameMaker""; ""Antenna ..."
4,11-2011.00,Advertising and Promotions Managers,73,"""Actuate BIRT""; ""AdRelevance""; ""Adobe Acrobat""..."
...,...,...,...,...
918,53-7071.00,Gas Compressor and Gas Pumping Station Operators,6,"""Computerized maintenance management system CM..."
919,53-7072.00,"Pump Operators, Except Wellhead Pumpers",7,"""Computerized maintenance management system CM..."
920,53-7073.00,Wellhead Pumpers,8,"""Microsoft Excel""; ""Microsoft Office software""..."
921,53-7081.00,Refuse and Recyclable Material Collectors,10,"""AMCS Platform""; ""Computerized maintenance man..."


# Next, I should look at how many JOB TITLES each SOFTWARE is associated with. 

# Also, how unique is the SOFTWARE among other SOFTWARE?

# Then, each job can be assigned a score for the uniqueness of the skills required. 

# Feel like I also saw a file breaking down each type of software with other stats, was that real?